# Data Quality — Setup Notebook

**Run this notebook once** to create the tables that power the validation system.

- `check_registry` — stores the model names, DAX expressions, and row-level fail thresholds you want to validate
- `check_baseline_config` — stores whether each check compares by MODEL baseline row or STATIC numeric value
- `validation_results` — stores the results of each run, including the fail threshold used for each row

After this setup, go to `02_data_quality_manage_checks_notebook` to add your first checks.

In [ ]:
# -- Step 1: Configuration -----------------------------------------------------
# Load shared settings from config.py in Lakehouse.
import sys
sys.path.append("/lakehouse/default/Files/code")

try:
    from config import LAKEHOUSE_NAME, SCHEMA_NAME, MAX_RETRY_ATTEMPTS, INITIAL_RETRY_DELAY, DAX_TIMEOUT_SECONDS, RUN_TABLE_MAINTENANCE, MAINTENANCE_DAY_UTC
except ImportError:
    # Fallback keeps the notebook runnable if config file is unavailable.
    LAKEHOUSE_NAME = "MyLakehouse"
    SCHEMA_NAME = "data_quality"
    MAX_RETRY_ATTEMPTS = 3
    INITIAL_RETRY_DELAY = 2
    DAX_TIMEOUT_SECONDS = 60
    RUN_TABLE_MAINTENANCE = False
    MAINTENANCE_DAY_UTC = 6

FULL_SCHEMA = f"{LAKEHOUSE_NAME}.{SCHEMA_NAME}"

In [ ]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {FULL_SCHEMA}")

# Minimal setup: create required Delta tables with the current schema only.
# No legacy backfill or migration logic is applied in this simplified version.

check_registry_ddl = f"""
CREATE TABLE IF NOT EXISTS {FULL_SCHEMA}.check_registry (
    check_name                 STRING  NOT NULL COMMENT 'Same name across all models so the job knows to compare them',
    model_name                 STRING  NOT NULL COMMENT 'Friendly label - display name only',
    workspace_id               STRING  NOT NULL COMMENT 'Fabric workspace GUID for stable identity',
    dataset_id                 STRING  NOT NULL COMMENT 'Semantic model GUID for stable identity',
    workspace_name             STRING  NOT NULL COMMENT 'Fabric workspace display name',
    dataset_name               STRING  NOT NULL COMMENT 'Semantic model / dataset display name',
    dax_expression             STRING  NOT NULL COMMENT 'DAX returning a single number, e.g. EVALUATE ROW("value", [Sales Amount])',
    run_frequency              STRING  NOT NULL COMMENT 'ONCE_PER_DAY (run max once per day) or MULTIPLE_PER_DAY (can run multiple times)',
    fail_delta_pct_threshold   DOUBLE  NOT NULL COMMENT 'Percent delta threshold for this row. Example: 0.5 means fail when absolute delta exceeds 0.5%',
    is_active                  BOOLEAN NOT NULL COMMENT 'Set to false to skip this row without deleting it',
    is_baseline                BOOLEAN NOT NULL COMMENT 'True for exactly one row per check_name - used as the comparison reference'
 )
USING DELTA
COMMENT 'Add one row per model per check - all rows sharing a check_name will be compared. PRIMARY KEY uses stable IDs.'
"""

check_baseline_config_ddl = f"""
CREATE TABLE IF NOT EXISTS {FULL_SCHEMA}.check_baseline_config (
    check_name             STRING  NOT NULL COMMENT 'Check identifier shared across models',
    baseline_mode          STRING  NOT NULL COMMENT 'MODEL uses is_baseline row, STATIC uses static_baseline_value',
    static_baseline_value  DOUBLE  COMMENT 'Required when baseline_mode = STATIC',
    updated_at             TIMESTAMP NOT NULL
 )
USING DELTA
COMMENT 'Per-check baseline mode configuration. MODEL uses check_registry.is_baseline; STATIC uses static number.'
"""

validation_results_ddl = f"""
CREATE TABLE IF NOT EXISTS {FULL_SCHEMA}.validation_results (
    run_id                     STRING    NOT NULL COMMENT 'UUID identifying this specific run',
    run_date                   DATE      NOT NULL,
    run_timestamp              TIMESTAMP NOT NULL,
    check_name                 STRING    NOT NULL,
    model_name                 STRING    NOT NULL,
    workspace_id               STRING    COMMENT 'Fabric workspace GUID copied from registry',
    dataset_id                 STRING    COMMENT 'Semantic model GUID copied from registry',
    workspace_name             STRING    COMMENT 'Workspace display name copied from registry',
    dataset_name               STRING    COMMENT 'Dataset display name copied from registry',
    result_value               DOUBLE    COMMENT 'Value returned by the DAX for this model',
    baseline_model             STRING    COMMENT 'The model flagged as is_baseline in check_registry - used as reference',
    baseline_value             DOUBLE    COMMENT 'Value returned by the DAX for the baseline model',
    absolute_delta             DOUBLE    COMMENT 'result_value minus baseline_value',
    delta_pct                  DOUBLE    COMMENT 'absolute_delta divided by baseline_value, as a percentage',
    fail_delta_pct_threshold   DOUBLE    COMMENT 'Percent delta threshold copied from check_registry for this row execution',
    status                     STRING    NOT NULL COMMENT 'PASS | FAIL | ERROR',
    error_message              STRING    COMMENT 'Populated only when status = ERROR'
 )
USING DELTA
PARTITIONED BY (run_date)
COMMENT 'Validation results written by the daily job - do not edit manually'
"""

def create_with_recovery(table_name: str, ddl_sql: str):
    spark.sql(ddl_sql)
    try:
        spark.sql(f"DESCRIBE DETAIL {table_name}")
    except Exception as e:
        msg = str(e).lower()
        if "delta_path_does_not_exist" in msg or "doesn't exist, or is not a delta table" in msg:
            print(f"Detected broken Delta metadata for {table_name}. Recreating table...")
            spark.sql(f"DROP TABLE IF EXISTS {table_name}")
            spark.sql(ddl_sql)
        else:
            raise RuntimeError(
                f"Failed to validate {table_name}. Check LAKEHOUSE_NAME/SCHEMA_NAME config. "
                f"Original error: {str(e)[:300]}"
            )

create_with_recovery(f"{FULL_SCHEMA}.check_registry", check_registry_ddl)
create_with_recovery(f"{FULL_SCHEMA}.check_baseline_config", check_baseline_config_ddl)
create_with_recovery(f"{FULL_SCHEMA}.validation_results", validation_results_ddl)

print(f"Tables ready in {FULL_SCHEMA}")
print("Setup complete with minimal table creation logic (no backfill/migration steps).")

## Verify

In [ ]:
from IPython.display import display

# Show both tables exist
tables = spark.sql(f"SHOW TABLES IN {FULL_SCHEMA}").toPandas()
display(tables)

required_tables = {"check_registry", "check_baseline_config", "validation_results"}

if "isTemporary" in tables.columns:
    temp_flag = tables["isTemporary"]
    if temp_flag.dtype == object:
        temp_flag = temp_flag.astype(str).str.strip().str.lower().isin(["true", "1", "yes"])
    else:
        temp_flag = temp_flag.fillna(False).astype(bool)
    tables_real = tables[~temp_flag]
else:
    tables_real = tables

name_col = None
if "tableName" in tables_real.columns:
    name_col = "tableName"
else:
    lowered = {str(c).strip().lower(): c for c in tables_real.columns}
    for candidate in ["tablename", "table_name", "name", "table"]:
        if candidate in lowered:
            name_col = lowered[candidate]
            break

if name_col is None:
    raise RuntimeError(
        "Could not locate table-name column in SHOW TABLES output. "
        f"Available columns: {', '.join(map(str, tables_real.columns.tolist()))}"
    )

found_tables = set(tables_real[name_col].astype(str).str.strip().str.lower().tolist())
missing_tables = sorted(required_tables - found_tables)

if missing_tables:
    raise RuntimeError(
        f"Missing required table(s) in {FULL_SCHEMA}: {', '.join(missing_tables)}"
    )

print(f"\n✓  Required tables created in {FULL_SCHEMA}")
print("   Next: go to 02_data_quality_manage_checks_notebook to add/update checks, including fail_delta_pct_threshold per row")

## Optional: Enable Delta Column Defaults
Run this section **after** setup completes if you want Delta-enforced defaults on table columns.
This enables the Delta table feature and sets defaults without requiring defaults in `CREATE TABLE`.

In [ ]:
# -- Optional Post-Setup Step: Enable and Apply Delta Column Defaults ----------
tables_to_enable = [
    f"{FULL_SCHEMA}.check_registry",
    f"{FULL_SCHEMA}.check_baseline_config",
    f"{FULL_SCHEMA}.validation_results",
]

for table_name in tables_to_enable:
    try:
        spark.sql(
            f"ALTER TABLE {table_name} SET TBLPROPERTIES('delta.feature.allowColumnDefaults' = 'supported')"
        )
        print(f"Enabled allowColumnDefaults on {table_name}")
    except Exception as e:
        print(f"Note: could not enable allowColumnDefaults on {table_name}: {str(e)[:250]}")

default_statements = [
    # check_registry defaults
    f"ALTER TABLE {FULL_SCHEMA}.check_registry ALTER COLUMN run_frequency SET DEFAULT 'ONCE_PER_DAY'",
    f"ALTER TABLE {FULL_SCHEMA}.check_registry ALTER COLUMN fail_delta_pct_threshold SET DEFAULT 0.01",
    f"ALTER TABLE {FULL_SCHEMA}.check_registry ALTER COLUMN is_active SET DEFAULT true",
    f"ALTER TABLE {FULL_SCHEMA}.check_registry ALTER COLUMN is_baseline SET DEFAULT false",

    # check_baseline_config defaults
    f"ALTER TABLE {FULL_SCHEMA}.check_baseline_config ALTER COLUMN baseline_mode SET DEFAULT 'MODEL'",
    f"ALTER TABLE {FULL_SCHEMA}.check_baseline_config ALTER COLUMN updated_at SET DEFAULT current_timestamp()",

    # validation_results defaults
    f"ALTER TABLE {FULL_SCHEMA}.validation_results ALTER COLUMN run_timestamp SET DEFAULT current_timestamp()",
]

for sql_stmt in default_statements:
    try:
        spark.sql(sql_stmt)
        print(f"Applied default: {sql_stmt}")
    except Exception as e:
        msg = str(e).lower()
        if "not supported" in msg or "allowcolumndefaults" in msg:
            raise RuntimeError(
                "Delta column defaults are not enabled for this table. "
                "Run the ALTER TABLE ... SET TBLPROPERTIES step first."
            )
        print(f"Note: could not apply default ({sql_stmt}): {str(e)[:250]}")

print("\nDefault configuration step complete.")
print("New rows inserted without explicit values will now use these defaults where supported.")